# Kaggriculture corrected-E replay processing

Replacement notebook for Issue #30. Regenerates canonical schema-v3 Parquet **from raw Kaggriculture 1.32.7 replay JSON only** for the new `E_CORRECTED_V1` BC lineage.

Pinned repository commit: `553269ec724e4bc29264e0d2a4f1cf11b124e452`.

Outputs go to `/kaggle/working/canonical-v3`. Do **not** migrate old processed Parquet. This notebook processes replays only; it does not train BC/PPO and does not build the Rust engine.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, base64, re
from kaggle_secrets import UserSecretsClient

REPO = Path('/kaggle/working/Kaggriculture')
BRANCH = 'codex/issue-30-corrected-e-history'
TARGET_SHA = '553269ec724e4bc29264e0d2a4f1cf11b124e452'
RAW_ROOT = Path('/kaggle/input/kaggriculture-raw-replays')
OUT_ROOT = Path('/kaggle/working/canonical-v3')
SOURCE_DATASET = 'kaggriculture-raw-replays'
PARTITIONS = [
    '2026-08-17',
    '2026-08-18',
    '2026-08-19',
    '2026-08-20',
    '2026-08-21',
]

print('repo target :', TARGET_SHA)
print('raw root    :', RAW_ROOT)
print('output root :', OUT_ROOT)
print('partitions  :', PARTITIONS)

## 1. Fresh clone and exact-version preflight

Uses the Kaggle `GITHUB_TOKEN` secret via a transient Git HTTP header. The token is never written into the remote URL. A fresh clone prevents an old checkout or local patch from contaminating preprocessing.

In [ ]:
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert token, 'Missing Kaggle secret GITHUB_TOKEN'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()

if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run([
    'git', '-c', f'http.extraHeader=Authorization: Basic {auth}',
    'clone', '--branch', BRANCH, '--single-branch',
    'https://github.com/BillXu21/Kaggriculture.git', str(REPO),
], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', TARGET_SHA], check=True)
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == TARGET_SHA, (actual, TARGET_SHA)

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO) + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
os.environ['PYTHONPATH'] = env['PYTHONPATH']

probe = subprocess.check_output([
    sys.executable, '-c',
    'from replay_daily.constants import ENGINE_VERSION,SCHEMA_VERSION; print(ENGINE_VERSION); print(SCHEMA_VERSION)'
], cwd=str(REPO), env=env, text=True).strip().splitlines()
assert probe[-2:] == ['1.32.7', '3'], probe

print('checked out:', actual)
print('engine     :', probe[-2])
print('schema     :', probe[-1])

## 2. Verify raw replay partitions

Issue #30 requires regeneration from raw replay JSON. Missing or empty partitions fail before any processing starts. If the attached Kaggle dataset uses another path, edit only `RAW_ROOT` above.

In [ ]:
assert RAW_ROOT.exists(), f'Missing raw replay dataset: {RAW_ROOT}'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

parts = []
for date in PARTITIONS:
    src = RAW_ROOT / date
    if not src.exists():
        available = sorted(p.name for p in RAW_ROOT.iterdir() if p.is_dir())
        raise FileNotFoundError(f'Missing {src}; available={available}')
    replay_files = sorted(p for p in src.glob('*.json') if p.name != 'manifest.csv')
    assert replay_files, f'No replay JSON files in {src}'
    manifest = src / 'manifest.csv'
    parts.append((date, src, len(replay_files), manifest))

print(f"{'partition':<12} {'replays':>9}  manifest")
for date, src, n, manifest in parts:
    print(f"{date:<12} {n:>9}  {'yes' if manifest.exists() else 'no'}")
print('total replays:', sum(x[2] for x in parts))

## 3. Regenerate canonical schema-v3 Parquet

Runs one independent extraction per date with `--on-version-mismatch fail`. A partition `manifest.csv`, when present, is passed through for score/provenance metadata.

In [ ]:
summary_re = re.compile(r'extracted\s+(\d+)\s+records\s+from\s+(\d+)\s+replay\(s\)')
results = []

for date, src, expected_replays, manifest in parts:
    out = OUT_ROOT / f'{date}.parquet'
    if out.exists():
        out.unlink()

    cmd = [
        sys.executable, '-m', 'replay_daily', 'extract',
        '--input', str(src),
        '--output', str(out),
        '--format', 'parquet',
        '--source-dataset', SOURCE_DATASET,
        '--partition-date', date,
        '--on-version-mismatch', 'fail',
    ]
    if manifest.exists():
        cmd += ['--manifest', str(manifest)]

    print('\n==', date, '==')
    proc = subprocess.run(
        cmd, cwd=str(REPO), env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print(proc.stdout, end='')
    if proc.returncode != 0:
        raise RuntimeError(f'Extraction failed for {date}: rc={proc.returncode}')

    m = summary_re.search(proc.stdout)
    assert m, f'Could not parse extraction summary for {date}'
    rows = int(m.group(1))
    parsed = int(m.group(2))
    assert rows > 0 and parsed > 0
    results.append((date, out, rows, parsed))

print('\nAll partition extractions completed.')

## 4. Validate outputs and corrected-E contract

Checks Arrow row counts, schema version 3, schema-validating `replay_daily inspect`, and the authoritative Issue #30 history helper: day 4 invalid, adjacent day 5 valid, legacy always invalid.

In [ ]:
import pyarrow.parquet as pq
from bc_manager.economics import E_HISTORY_CORRECTED_V1, E_HISTORY_LEGACY, previous_net_cash

manifest_rows = []
for date, out, reported_rows, parsed_replays in results:
    pf = pq.ParquetFile(out)
    assert pf.metadata.num_rows == reported_rows, (date, pf.metadata.num_rows, reported_rows)

    versions = set(pq.read_table(out, columns=['schema_version'])['schema_version'].to_pylist())
    assert versions == {3}, (date, versions)

    days = pq.read_table(out, columns=['day'])['day'].to_pylist()
    assert days

    inspect_probe = subprocess.run(
        [sys.executable, '-m', 'replay_daily', 'inspect', str(out)],
        cwd=str(REPO), env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    assert inspect_probe.returncode == 0, inspect_probe.stdout

    manifest_rows.append({
        'partition': date,
        'path': str(out),
        'records': reported_rows,
        'replays': parsed_replays,
        'day_min': min(days),
        'day_max': max(days),
        'size_bytes': out.stat().st_size,
    })

assert previous_net_cash(E_HISTORY_CORRECTED_V1, 4, 10000.0, None) == (0.0, False)
assert previous_net_cash(E_HISTORY_CORRECTED_V1, 5, 12500.0, (4, 10000.0)) == (2500.0, True)
assert previous_net_cash(E_HISTORY_LEGACY, 5, 12500.0, (4, 10000.0)) == (0.0, False)

print(f"{'partition':<12} {'records':>10} {'days':>8} {'MiB':>10}")
for r in manifest_rows:
    print(f"{r['partition']:<12} {r['records']:>10} {r['day_min']:>2}-{r['day_max']:<2} {r['size_bytes']/(1024**2):>10.1f}")
print('E-history contract smoke: OK')

## 5. Write processing manifest and BC handoff

Canonical rows themselves are history-neutral. The corrected BC adapter derives `E_CORRECTED_V1` from exact adjacent `(episode_id, seat, day)` starts, with manager day 4 explicitly invalid.

In [ ]:
processing_manifest = {
    'manifest_version': 1,
    'purpose': 'Issue #30 corrected-E BC replay processing',
    'repository': 'BillXu21/Kaggriculture',
    'repository_branch': BRANCH,
    'repository_sha': TARGET_SHA,
    'engine_version': '1.32.7',
    'canonical_schema_version': 3,
    'source_dataset': SOURCE_DATASET,
    'e_history_training_contract': 'E_CORRECTED_V1',
    'manager_start_day': 4,
    'note': 'Regenerated from raw replay JSON. Do not migrate old processed Parquet.',
    'outputs': manifest_rows,
}
manifest_path = OUT_ROOT / 'corrected_e_processing_manifest.json'
manifest_path.write_text(json.dumps(processing_manifest, indent=2, sort_keys=True))

training_command = '''cd /kaggle/working/Kaggriculture
python -m bc_manager.cli \
  /kaggle/working/canonical-v3/2026-08-17.parquet \
  /kaggle/working/canonical-v3/2026-08-18.parquet \
  /kaggle/working/canonical-v3/2026-08-19.parquet \
  /kaggle/working/canonical-v3/2026-08-20.parquet \
  /kaggle/working/canonical-v3/2026-08-21.parquet \
  --variant E \
  --e-history-version E_CORRECTED_V1 \
  --train-dates 2026-08-17,2026-08-18,2026-08-19,2026-08-20 \
  --val-dates 2026-08-21 \
  --min-score 2950 \
  --device cuda --amp on \
  --checkpoint-dir /kaggle/working/bc-v1-E-corrected
'''
(OUT_ROOT / 'CORRECTED_BC_TRAIN_COMMAND.txt').write_text(training_command)

print(json.dumps(processing_manifest, indent=2))
print('\nBC handoff command:\n')
print(training_command)

## Expected notebook output

```text
/kaggle/working/canonical-v3/
├── 2026-08-17.parquet
├── 2026-08-18.parquet
├── 2026-08-19.parquet
├── 2026-08-20.parquet
├── 2026-08-21.parquet
├── corrected_e_processing_manifest.json
└── CORRECTED_BC_TRAIN_COMMAND.txt
```

After all cells pass, publish/save the notebook output. The corrected-BC notebook should attach these generated outputs read-only instead of rerunning extraction. Old BC-E / P1 / P2 / P3 remain `E_LEGACY`.